# Supplementary Figure S8 — full-retraining sensitivity to the Exact-K budget

This lightweight notebook regenerates the publication figure from the completed
`12c_kappa_retraining` source table. It does not train a model and does not open
external data.

By default the notebook uses `results_v8/12c_kappa_retraining`, falling back
to the tracked publication source table in `source_data/12c_kappa_retraining`.
Inputs can be redirected with `INTERGATE_KAPPA_RESULTS_DIR`; outputs can be
redirected with `INTERGATE_MANUSCRIPT_FIGURE_DIR`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/intergate-mpl")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in (HERE, *HERE.parents)
        if (path / "intergate").is_dir()
    ),
    HERE,
)
configured_results = os.environ.get("INTERGATE_KAPPA_RESULTS_DIR")
if configured_results:
    RESULTS_DIR = Path(configured_results).expanduser().resolve()
else:
    candidates = (
        PROJECT_ROOT / "results_v8" / "12c_kappa_retraining",
        PROJECT_ROOT / "source_data" / "12c_kappa_retraining",
    )
    RESULTS_DIR = next(
        (path.resolve() for path in candidates if path.is_dir()),
        candidates[0].resolve(),
    )
OUTPUT_DIR = Path(
    os.environ.get(
        "INTERGATE_MANUSCRIPT_FIGURE_DIR",
        RESULTS_DIR / "figures",
    )
).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_TABLE = RESULTS_DIR / "Supplementary_Table_Kappa_retraining_sensitivity.csv"
if not SOURCE_TABLE.is_file():
    raise FileNotFoundError(f"Missing source table: {SOURCE_TABLE}")

summary = pd.read_csv(SOURCE_TABLE).sort_values("kappa").reset_index(drop=True)
expected_kappa = np.array([0.005, 0.007, 0.010, 0.015, 0.020])
if not np.allclose(summary["kappa"].to_numpy(float), expected_kappa):
    raise RuntimeError("The completed kappa grid is not the expected five-density grid.")

required = {
    "K",
    "macro_f1_mean",
    "macro_f1_std",
    "macro_f1_drop_mean_without_GNN",
    "macro_f1_drop_mean_without_message_passing",
    "mean_pairwise_jaccard",
    "nogueira_stability",
    "edge_overlap_coefficient_vs_primary",
    "strict_fraction_of_K",
    "giant_fraction",
    "fold_over_matched_expectation_at_least_one_endpoint",
    "fold_over_matched_expectation_both_endpoints",
    "specificity_margin",
}
missing = required - set(summary.columns)
if missing:
    raise RuntimeError(f"Missing required columns: {sorted(missing)}")

display(summary.loc[:, ["kappa", "K", "macro_f1_mean", "nogueira_stability",
                        "strict_fraction_of_K", "specificity_margin"]])


In [ ]:
BLUE = "#1565C0"
ORANGE = "#E56A00"
TEAL = "#0B7A75"
PURPLE = "#6A3D9A"
RED = "#C62828"
GREY = "#667085"
PALE_BLUE = "#EAF2FB"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "legend.frameon": False,
})

x = np.arange(len(summary), dtype=float)
primary_index = int(np.flatnonzero(np.isclose(summary["kappa"], 0.010))[0])
tick_labels = [
    f"{kappa:.3f}\nK={int(k_edges):,}"
    for kappa, k_edges in zip(summary["kappa"], summary["K"])
]


def format_axis(axis, *, zero_line=False, one_line=False, ylim=None):
    axis.axvspan(primary_index - 0.23, primary_index + 0.23,
                 color=PALE_BLUE, alpha=0.95, zorder=0)
    axis.axvline(primary_index, color=BLUE, linestyle=(0, (2, 2)),
                 linewidth=1.2, alpha=0.85, zorder=1)
    if zero_line:
        axis.axhline(0, color=GREY, linestyle="--", linewidth=0.9, alpha=0.8)
    if one_line:
        axis.axhline(1, color=GREY, linestyle="--", linewidth=0.9, alpha=0.8)
    axis.set_xticks(x, tick_labels)
    axis.set_xlabel(r"Sparsity fraction $\kappa$ and resulting Exact-$K$")
    axis.grid(axis="y", color="#D9DEE7", linewidth=0.8, alpha=0.85)
    axis.set_axisbelow(True)
    if ylim is not None:
        axis.set_ylim(*ylim)


fig, axes = plt.subplots(2, 3, figsize=(15.6, 8.8), constrained_layout=True)

axis = axes[0, 0]
axis.errorbar(
    x,
    summary["macro_f1_mean"],
    yerr=summary["macro_f1_std"],
    color=BLUE,
    marker="o",
    markersize=6,
    linewidth=2.1,
    capsize=4,
)
axis.set_title("A  Development predictive performance", loc="left")
axis.set_ylabel("Macro-F1, mean ± SD")
format_axis(axis, ylim=(0.8815, 0.8955))

axis = axes[0, 1]
axis.plot(x, summary["macro_f1_drop_mean_without_GNN"], color=TEAL,
          marker="o", linewidth=2.1, label="GNN neutralized")
axis.plot(x, summary["macro_f1_drop_mean_without_message_passing"],
          color=ORANGE, marker="s", linewidth=2.1,
          label="Message passing neutralized")
axis.set_title("B  Reliance on graph-derived information", loc="left")
axis.set_ylabel("Macro-F1 decrease")
axis.legend(loc="upper left")
format_axis(axis, zero_line=True, ylim=(-0.001, 0.0265))

axis = axes[0, 2]
axis.plot(x, summary["nogueira_stability"], color=BLUE, marker="o",
          linewidth=2.1, label="Nogueira stability")
axis.plot(x, summary["mean_pairwise_jaccard"], color=ORANGE, marker="s",
          linewidth=2.1, label="Mean edge Jaccard")
axis.plot(x, summary["edge_overlap_coefficient_vs_primary"], color=TEAL,
          marker="^", linewidth=2.1, label="Overlap with primary")
axis.set_title("C  Basal-like multiseed stability", loc="left")
axis.set_ylabel("Index")
axis.legend(loc="lower right")
format_axis(axis, ylim=(0.42, 1.025))

axis = axes[1, 0]
axis.plot(x, summary["strict_fraction_of_K"], color=PURPLE, marker="o",
          linewidth=2.1, label="Strict 5/5 fraction")
axis.plot(x, summary["giant_fraction"], color=TEAL, marker="^",
          linewidth=2.1, label="Consensus giant fraction")
axis.set_title("D  Basal-like recovery and cohesion", loc="left")
axis.set_ylabel("Fraction")
axis.legend(loc="upper left")
format_axis(axis, ylim=(0.10, 0.73))

axis = axes[1, 1]
axis.plot(
    x,
    summary["fold_over_matched_expectation_at_least_one_endpoint"],
    color=BLUE,
    marker="o",
    linewidth=2.1,
    label="At least one endpoint",
)
axis.plot(
    x,
    summary["fold_over_matched_expectation_both_endpoints"],
    color=ORANGE,
    marker="s",
    linewidth=2.1,
    label="Both endpoints",
)
axis.set_title("E  Basal-like signature recovery", loc="left")
axis.set_ylabel("Fold over candidate-matched expectation")
axis.legend(loc="upper right")
format_axis(axis, one_line=True, ylim=(0.75, 6.9))

axis = axes[1, 2]
axis.fill_between(x, 0, summary["specificity_margin"], color="#FDECEC", alpha=0.8)
axis.plot(x, summary["specificity_margin"], color=RED, marker="o", linewidth=2.1)
axis.set_title("F  Basal-like signature specificity", loc="left")
axis.set_ylabel("Log2-fold margin over best alternative")
for xpos, value in zip(x, summary["specificity_margin"]):
    if value < 0.02:
        axis.annotate(f"{value:.4f}", (xpos, value), xytext=(0, 7),
                      textcoords="offset points", ha="center", fontsize=8)
format_axis(axis, zero_line=True, ylim=(-0.08, 1.62))

fig.suptitle("Full-retraining sensitivity to the Exact-K budget",
             fontsize=18, fontweight="bold")
fig.text(
    0.5,
    -0.012,
    "Development only • five seeds per density • Phase 1 and Phase 2 retrained • only κ changed",
    ha="center",
    va="top",
    color=GREY,
    fontsize=9.5,
)

basename = OUTPUT_DIR / "Supplementary_Figure_S8_Kappa_retraining_sensitivity"
for extension in ("pdf", "png", "svg"):
    fig.savefig(
        basename.with_suffix(f".{extension}"),
        dpi=320 if extension == "png" else None,
        bbox_inches="tight",
        facecolor="white",
    )
plt.show()
print(f"Saved figure files to: {OUTPUT_DIR}")
